# 42_06 - RQ2: Attack Success Rate backdoor FedAvg vs robust-agg (Paper 4-2)**Tujuan:** ukur ASR backdoor (proporsi sampel ber-trigger -> target benign)untuk FedAvg vs aturan tangguh, pada f=10/20/30%. Diharapkan robust-agg menekan ASR.**Sumber tunggal:** `robust_agg.py` (trigger+ASR) + `../aws/fed_node.py`.**Output:** `fed_out42/backdoor_asr.csv`.> Jalankan setelah 42_01. JANGAN mengarang angka.

## 0. Setup

In [ ]:
import importlib, sys, subprocess
for m in ('numpy','pandas','scikit-learn'):
    if importlib.util.find_spec(m.replace('scikit-learn','sklearn')) is None:
        subprocess.run([sys.executable,'-m','pip','install','-q',m],check=True)
import os, numpy as np, pandas as pd
from sklearn.metrics import matthews_corrcoef
AWS_DIR=os.path.abspath(os.path.join('..','aws'))
import importlib.util as _u
_spec=_u.spec_from_file_location('fed_node', os.path.join(AWS_DIR,'fed_node.py'))
fed=_u.module_from_spec(_spec); _spec.loader.exec_module(fed)
sys.path.insert(0, os.path.abspath('.')); import robust_agg as ra
OUTDIR='fed_out42'; os.makedirs(OUTDIR, exist_ok=True)
ROUNDS=60; LOCAL_EPOCHS=3; LR=3e-3; SEED=42
print('setup 42_06 siap')

## 1. Muat partisi + run_fed

In [ ]:
def need(path):
    assert os.path.exists(path),'PRASYARAT GAGAL: %s - jalankan 42_01 dulu'%path
    return path
d=np.load(need(os.path.join(OUTDIR,'parts_K10.npz')),allow_pickle=True)
X=np.asarray(d['X'],float); y=np.asarray(d['y'],int); parts=list(d['parts']); K=int(d['K'])
g=np.load(need(os.path.join(OUTDIR,'global_test.npz')),allow_pickle=True)
Xg=np.asarray(g['X'],float); yg=np.asarray(g['y'],int)
def predict_with(W,mu,sd):
    def f(Z):
        out,_=fed.forward(W, fed.zscore_apply(Z,mu,sd)); return (out>=0.5).astype(int)
    return f
def run_fed(rule='fedavg', attack=None, mal_ids=(), gamma=10.0, bd_frac=0.5,
            rounds=ROUNDS, epochs=LOCAL_EPOCHS, lr=LR, seed=SEED):
    cli=[]
    for k in range(K):
        ii=parts[k]; Xk=X[ii].copy(); yk=y[ii].copy(); rng=np.random.RandomState(seed+k)
        if attack=='label_flip' and k in mal_ids: yk=ra.attack_label_flip(yk,rng)
        if attack=='backdoor'  and k in mal_ids: Xk,yk=ra.poison_backdoor_data(Xk,yk,bd_frac,rng)
        if len(yk): mu,sd=fed.zscore_fit(Xk)
        else: mu,sd=np.zeros(X.shape[1]),np.ones(X.shape[1])
        cli.append(dict(X=fed.zscore_apply(Xk,mu,sd),y=yk,n=len(yk)))
    Wg=fed.init_weights(seed); f_byz=max(1,len(mal_ids)) if mal_ids else 1
    for t in range(rounds):
        wl=[]; sizes=[]
        for k in range(K):
            if cli[k]['n']==0: continue
            Wl=fed.train_local(Wg,cli[k]['X'],cli[k]['y'],epochs=epochs,lr=lr,mu=0.0,W_global=Wg,seed=seed)
            if attack=='scale' and k in mal_ids: Wl=ra.attack_scale_weights(Wl,gamma=gamma)
            wl.append(Wl); sizes.append(cli[k]['n'])
        Wg,_=ra.aggregate(rule,wl,sizes,f=f_byz,beta=max(1,f_byz))
    mu_g,sd_g=fed.zscore_fit(X); pf=predict_with(Wg,mu_g,sd_g); yp=pf(Xg)
    mcc=round(matthews_corrcoef(yg,yp),4)
    asr=round(ra.attack_success_rate(pf,Xg,yg),4) if attack=='backdoor' else None
    return mcc,asr,Wg
RULES=['fedavg','krum','multi_krum','trimmed_mean','median']
print('K',K,'| run_fed siap | rules',RULES)

## 2. ASR backdoor: rule x fraksi penyerang

In [ ]:
frac_map={0.1:(0,),0.2:(0,1),0.3:(0,1,2)}
rows=[]
for rule in RULES:
    for frac,mal in frac_map.items():
        mcc,asr,_=run_fed(rule, attack='backdoor', mal_ids=mal)
        rows.append({'rule':rule,'frac':frac,'MCC':mcc,'ASR':asr})
        print(f'{rule:12s} f={int(frac*100)}% -> MCC {mcc} ASR {asr}')
dfa=pd.DataFrame(rows); dfa.to_csv(os.path.join(OUTDIR,'backdoor_asr.csv'),index=False)
import IPython.display as ipd; ipd.display(dfa.pivot(index='rule',columns='frac',values='ASR'))
print('=== SEL 2 (ASR backdoor) SELESAI ===')

## 3. Catatan untuk naskah- Hasil -> Tabel tab:asr (`paper-4-2.tex`). ASR rendah = pertahanan backdoor berhasil.- FedAvg diharapkan ASR tinggi; Krum/Median/Trimmed menekan ASR. Dilaporkan apa adanya.- Trigger & ASR dari `robust_agg.py` (parameter terpusat) -> konsisten.